# Transformer Model Training
This notebook demonstrates training a transformer model for lens performance prediction using PyTorch.

### Initial Setup

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import time
import os
import pandas as pd
import math
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
import numpy as np

# --- 1. SETUP & PATHS ---
# Detect GPU (Mac uses 'mps', NVIDIA uses 'cuda', else 'cpu')
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on: {device}")

current_folder = os.getcwd()
data_folder = os.path.join(current_folder, "Prime Lenses + Data", "LensDataExports")
summary_file = os.path.join(current_folder, "Prime Lenses + Data", "CSVExports", "file_lens_summary.csv")

print(f"Loading summary from: {summary_file}")
print(f"Loading lens data from: {data_folder}")

# Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 1e-4 
EPOCHS = 50

Training on: cpu
Loading summary from: /Users/thara/Documents/Fall 2025/AI:ML/2.156-Lens-Project/Prime Lenses + Data/CSVExports/file_lens_summary.csv
Loading lens data from: /Users/thara/Documents/Fall 2025/AI:ML/2.156-Lens-Project/Prime Lenses + Data/LensDataExports


In [52]:
# ==========================================
# 2. DATASET & HELPER FUNCTIONS
# ==========================================

def index_lens_files(root_dir, ignore_files=None):
    """Scans for .csv files, skipping any in 'ignore_files'."""
    if ignore_files is None: ignore_files = []
    path_map = {}
    print(f"Scanning {root_dir} for lens files...")
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".csv"):
                if file in ignore_files: continue
                lens_name = os.path.splitext(file)[0]
                full_path = os.path.join(root, file)
                path_map[lens_name] = full_path
    print(f"Found {len(path_map)} lens files.")
    return path_map

class LensPerformanceDataset(Dataset):
    def __init__(self, summary_file, lens_data_root, material_vocab=None):
        self.summary_df = pd.read_csv(summary_file)
        summary_filename = os.path.basename(summary_file)
        self.file_path_map = index_lens_files(lens_data_root, ignore_files=[summary_filename])
        # Only use these columns for lens surface tokens
        self.surface_columns = ['Surface', 'TypeName', 'Comment', 'Radius', 'Thickness', 'Material', 'SemiDiameter']
        self.numeric_features = ['Radius', 'Thickness', 'SemiDiameter']
        self.categorical_feature = 'Material'
        self.target_cols = ['Poly', 'Effective F/#']
        if material_vocab is None:
            self.material_vocab = self._build_vocab()
        else:
            self.material_vocab = material_vocab

    def _build_vocab(self):
        unique_materials = set(['Air'])
        print("Building Material Vocabulary...")
        for lens_name, file_path in self.file_path_map.items():
            try:
                df = pd.read_csv(file_path, usecols=[self.categorical_feature])
                unique_materials.update(df[self.categorical_feature].astype(str).unique())
            except: pass
        return {name: i for i, name in enumerate(sorted(unique_materials))}

    def __len__(self):
        return len(self.summary_df)

    def __getitem__(self, idx):
        row = self.summary_df.iloc[idx]
        lens_name = row['File Name']
        lookup_name = lens_name + "_LensData"
        if lookup_name not in self.file_path_map: return None
        try:
            lens_df = pd.read_csv(self.file_path_map[lookup_name], usecols=self.surface_columns)
            # Clean Inf and NaN values
            lens_df = lens_df.replace([np.inf, -np.inf], np.nan)
            lens_df = lens_df.fillna(0)
            # Numeric features
            lens_numeric = torch.tensor(lens_df[self.numeric_features].values, dtype=torch.float32)
            # Categorical (material)
            materials = lens_df[self.categorical_feature].astype(str).map(self.material_vocab).fillna(0)
            lens_material_ids = torch.tensor(materials.values, dtype=torch.long)
            # Targets
            targets = row[self.target_cols].values.astype(float)
            targets = np.nan_to_num(targets, nan=0.0)  # <--- This line fixes NaNs in targets
            target_tensor = torch.tensor(targets, dtype=torch.float32)
            return {'numeric_seq': lens_numeric, 'material_seq': lens_material_ids, 'targets': target_tensor}
        except Exception as e:
            print(f"Error loading {lookup_name}: {e}")
            return None
    
def lens_collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0: return None
    numeric_seqs = [item['numeric_seq'] for item in batch]
    material_seqs = [item['material_seq'] for item in batch]
    targets = [item['targets'] for item in batch]
    padded_numeric = pad_sequence(numeric_seqs, batch_first=True, padding_value=0.0)
    padded_materials = pad_sequence(material_seqs, batch_first=True, padding_value=0)
    mask = (padded_materials != 0) | (padded_numeric.abs().sum(dim=2) > 0)
    return {'numeric_seq': padded_numeric, 'material_seq': padded_materials, 'mask': mask, 'targets': torch.stack(targets)}

In [10]:
# ==========================================
# 3. TRANSFORMER MODEL
# ==========================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class LensTransformer(nn.Module):
    def __init__(self, num_materials, d_model=64, nhead=4, num_layers=3, output_dim=2):
        super().__init__()
        self.mat_embed = nn.Embedding(num_materials, 16)
        self.num_embed = nn.Linear(3, d_model - 16) 
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, output_dim)
        )
    def forward(self, numeric_seq, material_seq, src_key_padding_mask):
        mat_vecs = self.mat_embed(material_seq) 
        num_vecs = self.num_embed(numeric_seq)  
        x = torch.cat([num_vecs, mat_vecs], dim=2) 
        x = self.pos_encoder(x)
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        valid_mask = (~src_key_padding_mask).unsqueeze(-1).float()
        sum_embeddings = (x * valid_mask).sum(dim=1)
        num_valid = valid_mask.sum(dim=1)
        lens_vector = sum_embeddings / (num_valid + 1e-9)
        return self.head(lens_vector)

In [ ]:
# ==========================================
# DIAGNOSTIC: Count and inspect non-skipped batches
# ==========================================
processed_batches = 0
loader_iter = iter(train_loader)
for i in range(20):  # Check up to 20 batches
    try:
        batch = next(loader_iter)
        if batch is None:
            print(f"Batch {i}: None")
            continue
        if (
            torch.isnan(batch['numeric_seq']).any() or
            torch.isinf(batch['numeric_seq']).any() or
            torch.isnan(batch['material_seq']).any() or
            torch.isinf(batch['material_seq']).any() or
            torch.isnan(batch['targets']).any() or
            torch.isinf(batch['targets']).any() or
            batch['mask'].sum() == 0
        ):
            print(f"Batch {i}: Skipped due to NaN/Inf or empty mask")
            continue
        processed_batches += 1
        print(f"Batch {i}: PROCESSED")
        print("numeric_seq:", batch['numeric_seq'])
        print("material_seq:", batch['material_seq'])
        print("mask:", batch['mask'])
        print("targets:", batch['targets'])
        break  # Only print the first valid batch
    except StopIteration:
        print(f"Batch {i}: End of loader")
        break
    except Exception as e:
        print(f"Batch {i}: Exception {e}")
print(f"Total non-skipped batches found: {processed_batches}")




# ==========================================
# 4. TRAINING LOOP
# ==========================================

full_dataset = LensPerformanceDataset(summary_file, data_folder)
if len(full_dataset) == 0: raise ValueError("Dataset is empty! Check paths.")
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=lens_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=lens_collate_fn)
num_materials = len(full_dataset.material_vocab)
print(f"Vocab Size: {num_materials}")
model = LensTransformer(num_materials=num_materials, output_dim=2).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
print("\n--- Starting Training ---")
for epoch in range(50):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        if (
            torch.isnan(batch['numeric_seq']).any() or
            torch.isinf(batch['numeric_seq']).any() or
            torch.isnan(batch['material_seq']).any() or
            torch.isinf(batch['material_seq']).any() or
            torch.isnan(batch['targets']).any() or
            torch.isinf(batch['targets']).any() or
            batch['mask'].sum() == 0
        ):
            print("Skipping batch due to NaN/Inf or empty mask")
            continue
        numeric = batch['numeric_seq'].to(device)
        material = batch['material_seq'].to(device)
        mask = batch['mask'].to(device)
        targets = batch['targets'].to(device)
        optimizer.zero_grad()
        predictions = model(numeric, material, mask)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_train_loss = running_loss / len(train_loader)
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for batch in test_loader:
            if batch is None: continue
            numeric = batch['numeric_seq'].to(device)
            material = batch['material_seq'].to(device)
            mask = batch['mask'].to(device)
            targets = batch['targets'].to(device)
            loss = criterion(model(numeric, material, mask), targets)
            running_test_loss += loss.item()
    avg_test_loss = running_test_loss / len(test_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/50] Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")
print("Done! Saving model...")
torch.save(model.state_dict(), "lens_transformer_model.pth")

Batch 0: PROCESSED
numeric_seq: tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7927e+02,  1.4502e+01,  5.6760e+01],
         [-4.2324e+02,  3.0000e-01,  5.6760e+01],
         ...,
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 4.1000e+01,  4.6000e+00,  1.6820e+01],
         [ 1.9790e+02,  1.0000e-01,  1.6820e+01],
         ...,
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 2.1716e+02,  3.8200e+00,  2.8390e+01],
         [-3.3036e+03,  5.4700e+00,  2.8390e+01],
         ...,
         [ 0.0000e+00,  2.5000e+00,  2.3130e+01],
         [ 0.0000e+00,  1.0000e+00,  2.3130e+01],
         [ 0.0000e+00,  0.0000e+00,  2.1700e+01]],

        ...,

      

Model training complete. The transformer weights are saved to `lens_transformer_model.pth`.

In [61]:
# ==========================================
# DEBUGGING DATASET AND DATALOADER
# ==========================================
print("\n--- Dataset Debugging ---")
invalid_count = 0
valid_count = 0
for idx in range(len(full_dataset)):
    row = full_dataset.summary_df.iloc[idx]
    lens_name = row['File Name']+"_LensData"
    print(f"Index {idx}: Lens Name = {lens_name}", end=' ')
    if lens_name not in full_dataset.file_path_map:
        print("[MISSING FILE]")
        invalid_count += 1
        continue
    try:
        item = full_dataset[idx]
        if item is None:
            print("[ITEM NONE]")
            invalid_count += 1
        else:
            print("[VALID]")
            valid_count += 1
    except Exception as e:
        print(f"[EXCEPTION] {e}")
        invalid_count += 1
print(f"\nTotal valid items: {valid_count}")
print(f"Total invalid/missing items: {invalid_count}")

print("\n--- DataLoader Batch Debugging ---")
loader_iter = iter(train_loader)
for i in range(3):
    try:
        batch = next(loader_iter)
        if batch is None:
            print(f"Batch {i}: None")
        else:
            print(f"Batch {i}: Numeric shape {batch['numeric_seq'].shape}, Material shape {batch['material_seq'].shape}, Targets shape {batch['targets'].shape}")
    except StopIteration:
        print(f"Batch {i}: End of loader")
        break
    except Exception as e:
        print(f"Batch {i}: Exception {e}")


--- Dataset Debugging ---
Index 0: Lens Name = CH321571_Example01P_LensData [VALID]
Index 1: Lens Name = CH346706_Example01P_LensData [VALID]
Index 2: Lens Name = CN104101985_Example01P_LensData [VALID]
Index 3: Lens Name = CN106249387_Example02P_LensData [VALID]
Index 4: Lens Name = CN107255857_Example01P_LensData [VALID]
Index 5: Lens Name = CN107272156_Example01P_LensData [VALID]
Index 6: Lens Name = CN107272157_Example01P_LensData [VALID]
Index 7: Lens Name = CN110161666_Example02P_LensData [VALID]
Index 8: Lens Name = CN110501809_Example01P_LensData [VALID]
Index 9: Lens Name = CN110596863_Example01P_LensData [VALID]
Index 10: Lens Name = CN111965793_Example01P_LensData [VALID]
Index 11: Lens Name = CN205427291_Example02P_LensData [VALID]
Index 12: Lens Name = CN205720849_Example02P_LensData [VALID]
Index 13: Lens Name = CN206074892_Example01P_LensData [VALID]
Index 14: Lens Name = CN207216120_Example01P_LensData [VALID]
Index 15: Lens Name = CN209606699_Example01P_LensData [VALI

In [ ]:
# ==========================================
# BATCH AND TARGET DIAGNOSTICS BEFORE TRAINING
# ==========================================
loader_iter = iter(train_loader)
for i in range(3):
    try:
        batch = next(loader_iter)
        if batch is None:
            print(f"Batch {i}: None")
            continue
        print(f"Batch {i}: Numeric shape {batch['numeric_seq'].shape}, Material shape {batch['material_seq'].shape}, Targets shape {batch['targets'].shape}")
        print(f"Batch {i} targets: {batch['targets']}")
        # Check for NaNs/Infs
        for key in ['numeric_seq', 'material_seq', 'targets']:
            arr = batch[key]
            if torch.isnan(arr).any():
                print(f"Batch {i}: {key} contains NaNs!")
            if torch.isinf(arr).any():
                print(f"Batch {i}: {key} contains Infs!")
    except StopIteration:
        print(f"Batch {i}: End of loader")
        break
    except Exception as e:
        print(f"Batch {i}: Exception {e}")

Batch 0: Numeric shape torch.Size([16, 49, 3]), Material shape torch.Size([16, 49]), Targets shape torch.Size([16, 2])
Batch 0 targets: tensor([[0.7368, 2.9327],
        [0.0000, 1.8110],
        [1.5295, 3.0203],
        [0.9719, 2.1047],
        [0.0469, 3.8310],
        [0.9253, 2.9210],
        [2.7712, 1.4844],
        [0.5665, 3.7935],
        [0.0733, 4.1883],
        [1.7993, 2.0764],
        [0.8364, 3.6607],
        [0.6864, 3.6651],
        [0.1126, 3.6257],
        [0.1450, 6.5930],
        [0.0412, 3.1357],
        [0.0611, 4.1979]])
Batch 1: Numeric shape torch.Size([16, 31, 3]), Material shape torch.Size([16, 31]), Targets shape torch.Size([16, 2])
Batch 1 targets: tensor([[1.0364, 2.0742],
        [0.0099, 8.6654],
        [0.0719, 4.4003],
        [0.0347, 2.1397],
        [0.0820, 3.5477],
        [0.1071, 6.1636],
        [0.0782, 3.4951],
        [2.4181, 2.2040],
        [0.3853, 3.7747],
        [0.0000, 1.5197],
        [0.4091, 3.0932],
        [0.0312, 3.4958],